# 03 · Synthetic data, and the curation that makes it safe

**The decision this notebook supports:** where does training data come from when you have documents but no conversation logs?

This is the cold-start problem every enterprise AI project hits. Here Gemini generates candidate examples and the curation is written by hand, so you see exactly what curation removes.

**This notebook is optional and needs a `GEMINI_API_KEY`.** The lab already ships a curated SFT dataset (`data/generated/sft_train.json`, rebuilt offline by `scripts/build_session_sft.py`), so notebook 05 runs without this one. Files generated here go to `data/generated/gemini/` and never overwrite the shipped data.

The rule that matters: **generated data is a hypothesis until it is checked against ground truth.** A generator that invents a plausible range figure will teach your model to invent that figure, permanently. Curation is not a tidying step, it is the safety gate.

In [ ]:
# Load this project's shared paths, configuration, scoring, and file helpers.
# sys controls Python's import search path; adding '..' assumes the notebook folder
# is the working directory. Import the project root, not src itself, so src/secrets.py
# does not replace Python's own secrets module. warnings controls diagnostic messages;
# the existing filter hides warnings, which can also hide useful compatibility clues.
# json handles JSON data; it is imported here for later cells (unused in this setup).
# CFG is a nested dictionary: CFG["model"]["name"] reads its model-name setting.
import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])

## Connect to Gemini

The key is read from the `GEMINI_API_KEY` environment variable, or from `.env`, at call time (see `docs/SETUP.md`). It is never printed and never written to disk from here.

In [ ]:
# Create the Gemini SDK client through lab's helper, which reads the API key
# locally. CFG chooses the provider model name. generate_content sends a real
# network request and can consume API quota/cost; the reply checks connectivity.
# resp.text is the returned text, and strip() removes outer whitespace.
client = lab.gemini_client()
GEM = CFG["generation"]["gemini_model"]

resp = client.models.generate_content(model=GEM, contents="Reply with exactly: ok")
print("model:", GEM, "| reply:", resp.text.strip())

If that cell fails on the model name, list what your key can actually reach and update `configs/config.yaml`. Model availability changes.

In [ ]:
# List models exposed by the current API account to diagnose a model-name issue.
# getattr(object, name, default) safely reads an optional attribute.
# In this original code, 'or True' makes the if condition always true: every
# listed model is printed, not only models supporting generateContent.
for m in client.models.list():
    if "generateContent" in getattr(m, "supported_actions", []) or True:
        print(m.name)

## Generate instruction pairs

Three kinds, deliberately, because they teach different things:

1. **Grounded answers** to answerable questions. Teaches the format and the tone.
2. **Refusals** for unanswerable questions. Teaches the behaviour that matters commercially.
3. **Paraphrases** of the same question. Teaches robustness to how customers actually phrase things.

Temperature is high here, at 0.9, because we want *diversity*. That is the opposite of the judging setting later, which is 0.

In [ ]:
# Build a source-grounded generation prompt and a helper that requests JSON pairs.
# The first f-string inserts DOCS and STYLE now. Doubled braces {{n}} and {{topic}}
# remain literal placeholders so .format(...) can fill them separately per call.
# This supplies evidence to the generator; its output still requires review.
DOCS = "\n\n".join(p.read_text() for p in sorted(lab.RAW.glob("*.md")))
STYLE = (lab.RAW / "brand_guide.md").read_text()

GEN_PROMPT = f'''You are generating training data for a customer-service assistant for
Meridian Motors, a fictional Indian electric two-wheeler company.

Here are the ONLY documents that exist about this company:

{DOCS}

And here is the required voice:

{STYLE}

Generate {{n}} question-and-answer pairs about the topic: {{topic}}.

Rules:
- Every number in an answer MUST appear verbatim in the documents above. Never estimate.
- Answers are 1 to 3 sentences, plain, no marketing language.
- Vary how the question is phrased: some formal, some casual, some with typos.
- Return ONLY a JSON array of objects with keys "question" and "answer". No markdown fence.'''

# json parses JSON arrays into Python lists/dictionaries. re is Python's regular
# expression library; here it removes optional Markdown code fences from replies.
import json, re
# This function makes an API request each time it is called. Temperature controls
# sampling variety; max_output_tokens caps the whole response, not each answer.
# A response cut off by the cap can be invalid JSON and make json.loads fail.
def generate_pairs(topic, n=12):
    r = client.models.generate_content(
        model=GEM, contents=GEN_PROMPT.format(n=n, topic=topic),
        config={"temperature": CFG["generation"]["temperature"],
                "max_output_tokens": 4096})
    # The raw string r"..." keeps regex backslashes literal. ^ and $ anchor a line,
    # (?:json)? optionally matches the word json, and re.M makes anchors line-based.
    # This removes fences only; it does not validate facts or the response schema.
    txt = re.sub(r"^```(?:json)?|```$", "", r.text.strip(), flags=re.M).strip()
    return json.loads(txt)


In [ ]:

# Send a small real API request before the larger topic loop. Iterate over the
# returned dictionaries to inspect both questions and answers; the same-line for
# loop is normal Python syntax, with print as its repeated body.
sample = generate_pairs("battery capacity and range of each model", n=4)
for s in sample: print("Q:", s["question"], "\nA:", s["answer"], "\n")

In [ ]:
# Request multiple topics so the training pool covers more than one attribute.
# tqdm.auto adds progress feedback to the API loop. Each successful topic requests
# 12 pairs; dict(p, topic=t) copies a returned pair and attaches provenance metadata.
# += extends raw_pairs with the list produced by the comprehension.
TOPICS = [
    "battery capacity and certified range of each model",
    "charging times, both standard and fast charging",
    "on-road prices in Mumbai, Bengaluru and Delhi",
    "warranty periods and battery warranty distance limits",
    "service intervals and what the warranty excludes",
    "kerb weight, storage space and available colours",
    "top speed and motor power",
    "comparing two Meridian models against each other",
]

from tqdm.auto import tqdm
raw_pairs = []
for t in tqdm(TOPICS, desc="generating"):
    try:
        raw_pairs += [dict(p, topic=t) for p in generate_pairs(t, n=12)]
    # The broad exception handler lets later topics continue after a failed request.
    # It prints the exception class rather than raw provider text. The final count
    # therefore reports actual returned pairs, which may be below the requested total.
    except Exception as e:
        print("failed on", t, type(e).__name__)
print("generated:", len(raw_pairs))

## Curate: the step that decides whether this data is safe

Four checks, in order. Count what each one removes, because that count is the interesting result, not the surviving dataset.

The **fact check** is the important one. For every number in a generated answer, does that number appear in `facts.json`? If the generator invented a figure, it is caught here and not six weeks later by a customer.

In [ ]:
# Curate structure, duplicate questions, answer length, and suspicious numbers.
# re finds number-like substrings; a set stores unique allowed numeric values.
# This is a coarse filter, not a semantic fact check: a valid number can still be
# attached to the wrong scooter/attribute, and unsupported nonnumeric claims pass.
# Product model numbers and allowed small values also need manual interpretation.
import re
facts = lab.facts()["facts"]
known_numbers = set()
for f in facts:
    v = f["value"]
    if isinstance(v, (int, float)):
        known_numbers.add(float(v))

# The regex accepts digits with commas and an optional decimal portion.
# A list comprehension removes grouping commas and converts matches to floats.
def numbers_in(text):
    return [float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*(?:\.\d+)?", text)]

# Return two values: whether the numeric check passed and the suspect numbers.
# any(...) accepts a value near any known number; 1e-6 tolerates floating-point
# rounding. The explicit small-number whitelist is permissive, not source proof.
def fact_check(answer):
    nums = numbers_in(answer)
    if not nums: return True, []
    bad = [n for n in nums if not any(abs(n - k) < 1e-6 for k in known_numbers)
           and n not in (0,1,2,3,4,5,80,100)]   # allow small ordinals and percentages
    return len(bad) == 0, bad


In [ ]:

# Process pairs in order and record the first rejection reason for each.
# .get supplies a default for missing keys; strip removes outer whitespace.
# continue skips the remaining checks for that pair and advances the loop.
kept, dropped = [], {"duplicate":0, "too_long":0, "invented_number":0, "no_question":0}
seen = set()
for p in raw_pairs:
    q, a = p.get("question","").strip(), p.get("answer","").strip()
    if not q or not a: dropped["no_question"] += 1; continue
    # Normalize case and remove non-word characters for duplicate detection.
    # This catches surface duplicates, not paraphrases with the same meaning.
    key = re.sub(r"\W+","",q.lower())
    if key in seen: dropped["duplicate"] += 1; continue
    if len(a.split()) > 90: dropped["too_long"] += 1; continue
    ok, bad = fact_check(a)
    if not ok: dropped["invented_number"] += 1; continue
    # Only accepted questions enter seen, so a rejected earlier version does not
    # block a later acceptable version. Semicolons put two statements on one line.
    seen.add(key); kept.append({"question":q,"answer":a,"topic":p.get("topic")})

# The rejection rate measures these implemented filters, not the true factual
# error rate. max(len(raw_pairs), 1) avoids division by zero for an empty response pool.
print(f"generated {len(raw_pairs)} -> kept {len(kept)}")
for k,v in dropped.items(): print(f"  dropped {k:18s} {v}")
print(f"\nrejection rate: {100*(len(raw_pairs)-len(kept))/max(len(raw_pairs),1):.1f}%")

**Look at that rejection rate.** If it is above roughly 10%, your generator is hallucinating into your training set at a rate that would have been invisible without this check. That number is worth remembering: it is the concrete argument for why synthetic data needs curation, and it is a good interview answer.

In [ ]:
# Sanity-check the numeric filter with a deliberately incorrect claim.
# The returned bad list contains suspicious numeric tokens; it does not identify
# which model/attribute was contradicted. A passing filter would still need review.
ok, bad = fact_check("The Meridian Arc 110 has a range of 140 km and costs Rs 99,000.")
print("caught invented numbers:", bad)

## Add refusal examples

These are written deterministically rather than generated, because the *behaviour* must be exactly consistent. A generator producing 15 slightly different hedges teaches a fuzzy policy.

In [ ]:
# random is Python's pseudorandom-choice library. Random(42) creates a local,
# repeatable generator without changing every other use of randomness.
# Refusal questions come from the training-only refusal seed, never from evals/,
# so a later refusal score is not measured on questions the model trained on.
import json, random
rng = random.Random(42)
seed = json.loads((lab.GEN / "sft_session" / "policy_refusal_seed.json").read_text())
UNANS = [q for fam in seed if fam["kind"] == "refusal" for q in fam["questions"][:2]]
eval_questions = {q["question"] for name in ("dev", "dev_v1", "heldout") for q in lab.eval_set(name)}
assert not set(UNANS) & eval_questions, "refusal training questions must not overlap any evaluation set"
REFUSALS = [
 "I don't have that information in Meridian's documents. I can connect you with a dealer who can help.",
 "That isn't covered in the information I have. A Meridian dealer would be able to confirm it for you.",
 "I can't answer that from Meridian's official documents. Shall I put you in touch with a dealer?",
]
# For each question, choice picks one approved refusal wording. The topic field
# is provenance metadata; notebook 05 trains on question and answer, not this field.
refusal_pairs = [{"question": q, "answer": rng.choice(REFUSALS), "topic": "refusal"} for q in UNANS]
print(len(refusal_pairs), "refusal examples")
print(refusal_pairs[0])

## Split and save

The split is by **question**, with a fixed seed, and it never touches the held-out evaluation set. Training data and evaluation data must not overlap, or every number afterwards is meaningless.

In [ ]:
# Combine accepted grounded pairs and refusal pairs, then shuffle rows in place.
# The seeded rng makes the split repeatable for the same input ordering. Slicing
# [:n_val] selects validation rows; [n_val:] selects the remainder for training.
# This is a row split: related paraphrases/facts may appear on both sides.
# Output goes to data/generated/gemini/, so the shipped SFT dataset is never overwritten.
dataset = kept + refusal_pairs
rng.shuffle(dataset)
# Choose 12% for validation, with a minimum of eight rows. int truncates toward
# zero. With a very small dataset, this minimum can leave little or no training data.
n_val = max(8, int(0.12*len(dataset)))
train, val = dataset[n_val:], dataset[:n_val]

# json.dumps serializes each list to JSON text; write_text persists the files
# for notebook 05. The separate run artifact records counts, not validation
# that every retained statement is correct.
OUT = lab.GEN / "gemini"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "sft_train.json").write_text(json.dumps(train, indent=2))
(OUT / "sft_val.json").write_text(json.dumps(val, indent=2))
lab.save_run("05_data_generation", {
    "generated": len(raw_pairs), "kept": len(kept), "dropped": dropped,
    "refusals": len(refusal_pairs), "train": len(train), "val": len(val)})
print(f"train {len(train)} | val {len(val)} | saved to {OUT.relative_to(lab.ROOT)}/")
print("To train notebook 05 on these, copy them over data/generated/sft_train.json and sft_val.json.")

## Also build the continued-pretraining corpus

CPT does not use question-and-answer pairs. It uses **raw text**, and the model learns by predicting the next token across it. So the corpus here is the documents themselves, repeated and shuffled into chunks.

In [ ]:
# Prepare the original plain-text CPT input by joining sorted source documents.
# This cell writes one copy of the documents; it does not itself repeat, shuffle,
# or tokenize them. Notebook 04 trains on the shipped data/generated/cpt_session corpus instead.
# The printed count is whitespace-separated words, not model tokens.
cpt_text = "\n\n".join(p.read_text() for p in sorted(lab.RAW.glob("*.md")))
(lab.GEN / "gemini").mkdir(parents=True, exist_ok=True)
(lab.GEN / "gemini" / "cpt_corpus.txt").write_text(cpt_text)
print("CPT corpus:", len(cpt_text.split()), "words")

## What you should be able to say now

- What a curation step must remove before generated data is safe to train on.
- Your measured rejection rate, and why an uncurated synthetic set is dangerous rather than merely noisy.
- Why refusal examples are written rather than generated.

**Next:** notebook 04 runs continued pretraining. 